# SafeSynth FLUX.2 v2 method diagnostic

**Diagnostic only.** This notebook uses four Train-only inputs. It does not read Test, compute H4 AUC, reopen M13, or start Phase 2. Use an L4 GPU runtime.

In [ ]:
%pip install -q diffusers==0.39.0 transformers==5.14.1 accelerate==1.14.0 huggingface-hub==1.24.0 safetensors==0.7.0

In [ ]:
import json, os, platform, shutil, sys, time
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageDraw
from diffusers import Flux2KleinInpaintPipeline
from google.colab import files

assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > L4 GPU first.'
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

Upload `flux2_v2_colab_diagnostic_inputs.zip` when prompted.

In [ ]:
uploaded = files.upload()
zip_names = [name for name in uploaded if name.endswith('.zip')]
assert len(zip_names) == 1, f'Expected exactly one zip, got: {zip_names}'
input_root = Path('/content/flux2_v2_inputs')
input_root.mkdir(parents=True, exist_ok=True)
shutil.unpack_archive(zip_names[0], input_root)
manifest_paths = list(input_root.rglob('manifest.json'))
assert len(manifest_paths) == 1, manifest_paths
bundle_root = manifest_paths[0].parent
manifest = json.loads(manifest_paths[0].read_text(encoding='utf-8'))
assert manifest['diagnostic_only'] is True
assert manifest['source_split'] == 'train_only'
assert manifest['final_h4_auc_computed'] is False
print('Loaded cases:', len(manifest['cases']))

In [ ]:
model = manifest['model']
pipe = Flux2KleinInpaintPipeline.from_pretrained(
    model['repo_id'],
    revision=model['revision'],
    torch_dtype=torch.bfloat16,
)
pipe.enable_model_cpu_offload()
print('Loaded:', model['repo_id'], model['revision'])

The three comparisons were fixed before seeing these outputs. Do not add variants after running this cell.

In [ ]:
PROMPT = (
    'Photorealistic construction-site photograph. Integrate the existing pasted '
    'safety hard hat at its current location. Preserve its color, shape, viewing '
    'angle, scale, and center exactly. Modify only the masked boundary so lighting, '
    'focus, noise, and occlusion match the scene. Do not add another person, head, '
    'helmet, text, logo, or object.'
)
VARIANTS = [
    {'name': 'v1_reference_strength_085', 'strength': 0.85, 'use_reference': True},
    {'name': 'reference_strength_055', 'strength': 0.55, 'use_reference': True},
    {'name': 'no_reference_strength_055', 'strength': 0.55, 'use_reference': False},
]
result_root = Path('/content/flux2_v2_diagnostic_results')
result_root.mkdir(parents=True, exist_ok=True)
run_records = []

for case in manifest['cases']:
    case_dir = bundle_root / f"case_{case['canonical_contact_sheet_cell']:02d}"
    draft = Image.open(case_dir / 'draft.png').convert('RGB')
    mask = Image.open(case_dir / 'edit_mask.png').convert('L')
    reference = Image.open(case_dir / 'reference.png').convert('RGB')
    editable = np.asarray(mask) > 0
    for variant in VARIANTS:
        kwargs = dict(
            prompt=PROMPT,
            image=draft,
            mask_image=mask,
            padding_mask_crop=48,
            strength=variant['strength'],
            num_inference_steps=50,
            guidance_scale=4.0,
            generator=torch.Generator(device='cpu').manual_seed(case['generative_seed']),
            output_type='pil',
        )
        if variant['use_reference']:
            kwargs['image_reference'] = reference
        started = time.perf_counter()
        generated = pipe(**kwargs).images[0].convert('RGB')
        generated_array = np.asarray(generated.resize(draft.size), dtype=np.uint8)
        output_array = np.asarray(draft, dtype=np.uint8).copy()
        output_array[editable] = generated_array[editable]
        output = Image.fromarray(output_array)
        case_output_dir = result_root / case_dir.name
        case_output_dir.mkdir(parents=True, exist_ok=True)
        output_path = case_output_dir / f"{variant['name']}.png"
        output.save(output_path)
        elapsed = time.perf_counter() - started
        run_records.append({
            'case': case_dir.name,
            'sample_id': case['sample_id'],
            'variant': variant,
            'seconds': elapsed,
            'output': str(output_path.relative_to(result_root)),
        })
        print(case_dir.name, variant['name'], f'{elapsed:.1f}s')

In [ ]:
run_manifest = {
    'diagnostic_only': True,
    'final_h4_auc_computed': False,
    'gpu': torch.cuda.get_device_name(0),
    'model': model,
    'platform': platform.platform(),
    'python': sys.version,
    'torch': torch.__version__,
    'variants': VARIANTS,
    'runs': run_records,
}
(result_root / 'run_manifest.json').write_text(
    json.dumps(run_manifest, indent=2, sort_keys=True) + '\n', encoding='utf-8'
)

rows = len(manifest['cases'])
cols = 3 + len(VARIANTS)
size = 240
sheet = Image.new('RGB', (cols * size, rows * (size + 24)), 'white')
for row, case in enumerate(manifest['cases']):
    case_dir = bundle_root / f"case_{case['canonical_contact_sheet_cell']:02d}"
    panels = [
        ('DRAFT', Image.open(case_dir / 'draft.png').convert('RGB')),
        ('MASK', Image.open(case_dir / 'edit_mask.png').convert('RGB')),
        ('REFERENCE', Image.open(case_dir / 'reference.png').convert('RGB')),
    ]
    panels.extend(
        (variant['name'], Image.open(result_root / case_dir.name / f"{variant['name']}.png").convert('RGB'))
        for variant in VARIANTS
    )
    for col, (label, panel) in enumerate(panels):
        panel = panel.resize((size, size), Image.Resampling.LANCZOS)
        ImageDraw.Draw(panel).rectangle((0, 0, size, 19), fill='black')
        ImageDraw.Draw(panel).text((4, 3), label, fill='white')
        sheet.paste(panel, (col * size, row * (size + 24)))
    ImageDraw.Draw(sheet).text(
        (4, row * (size + 24) + size + 5),
        f"{case_dir.name} | {case['sample_id']}", fill='black'
    )
sheet.save(result_root / 'comparison_sheet.png', optimize=True)
display(sheet)

In [ ]:
archive = shutil.make_archive('/content/flux2_v2_diagnostic_results', 'zip', result_root)
print('Download:', archive)
files.download(archive)